# Orbital Elements and State Vector Conversion

This notebook bridges:
- Cartesian state vectors (r, v)
- Classical Keplerian elements (a, e, i, RAAN, omega, nu)
- Equinoctial elements

The flow is pedagogical: derivation summary, implementation, validation, singularity behavior, and geometric interpretation.

## 1) Imports and plotting setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

np.set_printoptions(precision=8, suppress=True)

## 2) Cartesian to Keplerian

Given gravitational parameter mu:
- Specific angular momentum vector: h = r x v
- Node vector: n = k x h (k is +z unit vector)
- Eccentricity vector: e_vec = (v x h)/mu - r/|r|
- Specific orbital energy: eps = v^2/2 - mu/|r|
- Semi-major axis: a = -mu/(2*eps) for non-parabolic orbits

Singular cases in classical elements:
- Circular (e -> 0): perigee direction is undefined, so omega and nu are not unique.
- Equatorial (i -> 0): line of nodes is undefined, so RAAN is not unique.

In [ ]:
def cartesian_to_keplerian(r, v, mu, tol=1e-10):
    r = np.asarray(r, dtype=float)
    v = np.asarray(v, dtype=float)

    r_norm = np.linalg.norm(r)
    v_norm = np.linalg.norm(v)

    h_vec = np.cross(r, v)
    h = np.linalg.norm(h_vec)

    k_hat = np.array([0.0, 0.0, 1.0])
    n_vec = np.cross(k_hat, h_vec)
    n = np.linalg.norm(n_vec)

    e_vec = np.cross(v, h_vec) / mu - r / r_norm
    e = np.linalg.norm(e_vec)

    energy = 0.5 * v_norm**2 - mu / r_norm
    if abs(energy) < tol:
        a = np.inf
    else:
        a = -mu / (2.0 * energy)

    i = np.arccos(np.clip(h_vec[2] / h, -1.0, 1.0))

    if n > tol:
        RAAN = np.arctan2(n_vec[1], n_vec[0]) % (2 * np.pi)
    else:
        RAAN = np.nan

    if n > tol and e > tol:
        omega = np.arccos(np.clip(np.dot(n_vec, e_vec) / (n * e), -1.0, 1.0))
        if e_vec[2] < 0:
            omega = 2 * np.pi - omega
    else:
        omega = np.nan

    if e > tol:
        nu = np.arccos(np.clip(np.dot(e_vec, r) / (e * r_norm), -1.0, 1.0))
        if np.dot(r, v) < 0:
            nu = 2 * np.pi - nu
    else:
        nu = np.nan

    # Auxiliary longitudes that remain useful around singular cases
    if n > tol:
        u = np.arctan2(np.dot(np.cross(n_vec, r), h_vec) / (n * h * r_norm), np.dot(n_vec, r) / (n * r_norm)) % (2*np.pi)
    else:
        u = np.nan
    l_true = np.arctan2(r[1], r[0]) % (2*np.pi)

    return {
        "h_vec": h_vec,
        "e_vec": e_vec,
        "a": a,
        "e": e,
        "i": i,
        "RAAN": RAAN,
        "omega": omega,
        "nu": nu,
        "u_arg_latitude": u,
        "l_true_longitude": l_true,
    }

## 3) Keplerian to Cartesian

Perifocal-frame formulas:
- p = a*(1 - e^2)
- r_pqw = [p/(1+e*cos(nu))] * [cos(nu), sin(nu), 0]
- v_pqw = sqrt(mu/p) * [-sin(nu), e+cos(nu), 0]

Then rotate into inertial coordinates using R3(RAAN) * R1(i) * R3(omega).

In [ ]:
def R3(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=float)


def R1(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]], dtype=float)


def keplerian_to_cartesian(a, e, i, RAAN, omega, nu, mu):
    p = a * (1 - e**2)

    r_pqw = (p / (1 + e * np.cos(nu))) * np.array([np.cos(nu), np.sin(nu), 0.0])
    v_pqw = np.sqrt(mu / p) * np.array([-np.sin(nu), e + np.cos(nu), 0.0])

    Q_pqw_to_eci = R3(RAAN) @ R1(i) @ R3(omega)
    r_eci = Q_pqw_to_eci @ r_pqw
    v_eci = Q_pqw_to_eci @ v_pqw
    return r_eci, v_eci

## 4) Validation with a random elliptical orbit

Process:
1. Randomly generate an elliptical orbit.
2. Convert Keplerian -> Cartesian.
3. Convert Cartesian -> Keplerian.
4. Print absolute differences.

In [ ]:
mu_earth = 398600.4418  # km^3/s^2
rng = np.random.default_rng(7)

a0 = rng.uniform(7000, 30000)
e0 = rng.uniform(0.05, 0.75)
i0 = np.deg2rad(rng.uniform(5, 175))
RAAN0 = np.deg2rad(rng.uniform(0, 360))
omega0 = np.deg2rad(rng.uniform(0, 360))
nu0 = np.deg2rad(rng.uniform(0, 360))

r0, v0 = keplerian_to_cartesian(a0, e0, i0, RAAN0, omega0, nu0, mu_earth)
kep = cartesian_to_keplerian(r0, v0, mu_earth)

print("Original elements:")
print(f"a     = {a0:.8f} km")
print(f"e     = {e0:.12f}")
print(f"i     = {np.rad2deg(i0):.8f} deg")
print(f"RAAN  = {np.rad2deg(RAAN0):.8f} deg")
print(f"omega = {np.rad2deg(omega0):.8f} deg")
print(f"nu    = {np.rad2deg(nu0):.8f} deg")

print()
print("Recovered elements:")
print(f"a     = {kep['a']:.8f} km")
print(f"e     = {kep['e']:.12f}")
print(f"i     = {np.rad2deg(kep['i']):.8f} deg")
print(f"RAAN  = {np.rad2deg(kep['RAAN']):.8f} deg")
print(f"omega = {np.rad2deg(kep['omega']):.8f} deg")
print(f"nu    = {np.rad2deg(kep['nu']):.8f} deg")

wrap = lambda x: (x + np.pi) % (2*np.pi) - np.pi

diffs = {
    "|delta a| (km)": abs(kep['a'] - a0),
    "|delta e|": abs(kep['e'] - e0),
    "|delta i| (deg)": abs(np.rad2deg(wrap(kep['i'] - i0))),
    "|delta RAAN| (deg)": abs(np.rad2deg(wrap(kep['RAAN'] - RAAN0))),
    "|delta omega| (deg)": abs(np.rad2deg(wrap(kep['omega'] - omega0))),
    "|delta nu| (deg)": abs(np.rad2deg(wrap(kep['nu'] - nu0))),
}

print()
print("Numerical differences:")
for k, val in diffs.items():
    print(f"{k:18s}: {val:.3e}")

## 5) Singularity demonstrations

### Case 1: Circular orbit (e approximately 0)
Perigee direction is undefined, so omega and nu become undefined as separate angles.

### Case 2: Equatorial orbit (i approximately 0)
Line of nodes is undefined, so RAAN becomes undefined.

In [ ]:
# Case 1: nearly circular, non-equatorial
a_c, e_c = 12000.0, 1e-12
i_c, RAAN_c, omega_c, nu_c = np.deg2rad(45), np.deg2rad(80), np.deg2rad(30), np.deg2rad(120)
r_c, v_c = keplerian_to_cartesian(a_c, e_c, i_c, RAAN_c, omega_c, nu_c, mu_earth)
kep_c = cartesian_to_keplerian(r_c, v_c, mu_earth)

print("Case 1: Circular-like orbit")
print(f"e = {kep_c['e']:.3e}")
print(f"RAAN  = {np.rad2deg(kep_c['RAAN']):.6f} deg (defined)")
print(f"omega = {kep_c['omega']} (undefined -> NaN expected)")
print(f"nu    = {kep_c['nu']} (undefined -> NaN expected)")
print(f"true longitude = {np.rad2deg(kep_c['l_true_longitude']):.6f} deg (defined)")

# Case 2: nearly equatorial, non-circular
a_e, e_e = 15000.0, 0.2
i_e, RAAN_e, omega_e, nu_e = np.deg2rad(1e-12), np.deg2rad(40), np.deg2rad(60), np.deg2rad(90)
r_e, v_e = keplerian_to_cartesian(a_e, e_e, i_e, RAAN_e, omega_e, nu_e, mu_earth)
kep_e = cartesian_to_keplerian(r_e, v_e, mu_earth)

print()
print("Case 2: Equatorial-like orbit")
print(f"i = {np.rad2deg(kep_e['i']):.3e} deg")
print(f"RAAN  = {kep_e['RAAN']} (undefined -> NaN expected)")
print(f"omega = {kep_e['omega']} (undefined in this implementation when RAAN is undefined)")
print(f"nu    = {np.rad2deg(kep_e['nu']):.6f} deg (still recoverable from e-vector)")
print(f"true longitude = {np.rad2deg(kep_e['l_true_longitude']):.6f} deg (defined)")

## 6) Equinoctial elements

Use equinoctial set (a, h, k, p, q, lambda):
- h = e * sin(RAAN + omega)
- k = e * cos(RAAN + omega)
- p = tan(i/2) * sin(RAAN)
- q = tan(i/2) * cos(RAAN)
- lambda = RAAN + omega + nu

This representation avoids the separate RAAN/omega singular decomposition for circular or equatorial limits.

In [ ]:
def classical_to_equinoctial(a, e, i, RAAN, omega, nu):
    h = e * np.sin(RAAN + omega)
    k = e * np.cos(RAAN + omega)
    p = np.tan(i / 2.0) * np.sin(RAAN)
    q = np.tan(i / 2.0) * np.cos(RAAN)
    lam = (RAAN + omega + nu) % (2*np.pi)
    return {"a": a, "h": h, "k": k, "p": p, "q": q, "lambda": lam}

E_nom = classical_to_equinoctial(a0, e0, i0, RAAN0, omega0, nu0)
E_cir = classical_to_equinoctial(a_c, e_c, i_c, RAAN_c, omega_c, nu_c)
E_eq = classical_to_equinoctial(a_e, e_e, i_e, RAAN_e, omega_e, nu_e)

print("Nominal equinoctial:", E_nom)
print()
print("Circular-like equinoctial:", E_cir)
print("Finite?", all(np.isfinite(list(E_cir.values()))))
print()
print("Equatorial-like equinoctial:", E_eq)
print("Finite?", all(np.isfinite(list(E_eq.values()))))

## 7) Orbit geometry plot in 3D

For the nominal test orbit, plot:
- orbit path
- line of nodes
- perigee direction
- angular momentum direction

In [ ]:
def sample_orbit_points(a, e, i, RAAN, omega, mu, N=400):
    nus = np.linspace(0, 2*np.pi, N)
    pts = np.array([keplerian_to_cartesian(a, e, i, RAAN, omega, nu, mu)[0] for nu in nus])
    return pts

pts = sample_orbit_points(a0, e0, i0, RAAN0, omega0, mu_earth, N=600)

h_vec = kep['h_vec']
h_hat = h_vec / np.linalg.norm(h_vec)
e_vec = kep['e_vec']
e_hat = e_vec / (np.linalg.norm(e_vec) + 1e-16)

n_vec = np.cross(np.array([0.0, 0.0, 1.0]), h_vec)
if np.linalg.norm(n_vec) > 1e-12:
    n_hat = n_vec / np.linalg.norm(n_vec)
else:
    n_hat = np.array([1.0, 0.0, 0.0])

scale = np.linalg.norm(pts, axis=1).max()

fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(111, projection='3d')

ax.plot(pts[:,0], pts[:,1], pts[:,2], label='Orbit', lw=2)

node_line = np.vstack([-n_hat*scale, n_hat*scale])
ax.plot(node_line[:,0], node_line[:,1], node_line[:,2], '--', lw=2, label='Line of nodes')

ax.quiver(0, 0, 0, *(e_hat*scale), color='tab:red', linewidth=2, label='Perigee direction')
ax.quiver(0, 0, 0, *(h_hat*scale), color='tab:green', linewidth=2, label='Angular momentum')

plane = np.linspace(-scale, scale, 2)
X, Y = np.meshgrid(plane, plane)
Z = np.zeros_like(X)
ax.plot_surface(X, Y, Z, alpha=0.08, color='gray')

ax.set_xlabel('x [km]')
ax.set_ylabel('y [km]')
ax.set_zlabel('z [km]')
ax.set_title('Orbital Geometry in Inertial Frame')
ax.legend(loc='upper left')
ax.set_box_aspect([1, 1, 1])
plt.tight_layout()
plt.show()

## 8) Why equinoctial elements avoid singularities

Classical elements isolate node and perigee directions; those directions are geometrically ambiguous in equatorial and circular limits.

Equinoctial elements combine orientation and shape into smooth variables:
- (h, k) remain well-behaved as e -> 0.
- (p, q) remain well-behaved as i -> 0.
- lambda remains meaningful where RAAN and omega individually are not.

Why mission design software prefers them:
- better numerical conditioning near near-circular/near-equatorial trajectories
- fewer branch conditions for singular edge cases
- improved robustness for optimization, estimation, and long-run propagation workflows

## 9) Comparison table: classical vs equinoctial

| Aspect | Classical Keplerian | Equinoctial |
|---|---|---|
| Element set | (a, e, i, RAAN, omega, nu) | (a, h, k, p, q, lambda) |
| Circular limit (e -> 0) | omega/nu become singular individually | smooth using (h, k, lambda) |
| Equatorial limit (i -> 0) | RAAN singular | smooth using (p, q) |
| Intuition | highly geometric and intuitive | slightly less intuitive at first |
| Numerical robustness near singular cases | moderate | generally better |
| Usage in mission software | common | very common, especially in robust solvers |